# Train Colosseum from a dataset ZIP in Google Drive

Run this notebook in [Google Colab](https://colab.research.google.com/github/Aqshalikhsan/Colosseum-Microcontroller/blob/main/notebooks/Train_Colosseum.ipynb). Select a GPU runtime first. The notebook reads a YOLO object-detection ZIP from Google Drive, extracts the dataset in Colab, and writes checkpoints and metrics back to Drive. Colosseum trains one target class per checkpoint from scratch; the included UC3M weights are not used here.


## 1. Install the framework

The PyPI distribution is `colosseum-microcontroller`; the Python import is `colosseum`.


In [ ]:
%pip install -q --upgrade colosseum-microcontroller
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())


## 2. Connect Google Drive and extract the dataset ZIP

Upload **one YOLO object-detection ZIP** to any folder in My Drive. Set `ZIP_PATH` below to its location. The ZIP can contain a wrapper folder, a Roboflow-style layout such as `train/images`, `train/labels`, `valid/images`, `valid/labels`, or a layout such as `images/train`, `labels/train`, `images/val`, `labels/val`. The notebook finds the training and validation folders, checks for images and YOLO `.txt` labels, and builds a `data.yaml` with the correct extracted paths. It also reads class names from the ZIP's `data.yaml` when available.

YOLO detection labels use one line per box: `class_id center_x center_y width height`, with coordinates from 0 to 1. COCO JSON, VOC XML, classification, and segmentation annotations need conversion to YOLO detection labels before training.

Extraction uses temporary Colab storage. The source ZIP and all training results stay in Drive. The mount prompt asks you to authorize Drive access.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import tempfile
from pathlib import Path
from zipfile import ZipFile
import yaml

DRIVE_ROOT = Path('/content/drive/MyDrive')
ZIP_PATH = DRIVE_ROOT / 'datasets' / 'my_dataset.zip'  # Change to your ZIP in My Drive.
OUTPUT_DIR = DRIVE_ROOT / 'Colosseum'                  # Checkpoints and metrics go here.
LOCAL_DATA_DIR = Path('/content')                      # Temporary extraction in Colab.

ZIP_PATH = ZIP_PATH.resolve()
OUTPUT_DIR = OUTPUT_DIR.resolve()
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f'Dataset ZIP not found: {ZIP_PATH}')
if ZIP_PATH.suffix.lower() != '.zip':
    raise ValueError(f'Expected a .zip dataset: {ZIP_PATH}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR = Path(tempfile.mkdtemp(prefix='colosseum_dataset_', dir=LOCAL_DATA_DIR))

with ZipFile(ZIP_PATH) as archive:
    for member in archive.infolist():
        target = (EXTRACT_DIR / member.filename).resolve()
        if not target.is_relative_to(EXTRACT_DIR):
            raise ValueError(f'Unsafe ZIP path: {member.filename}')
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError(f'ZIP contains a symbolic link: {member.filename}')
    archive.extractall(EXTRACT_DIR)

def find_layouts(extracted):
    layouts = []
    for train_dir in extracted.rglob('train'):
        if not train_dir.is_dir():
            continue
        if train_dir.parent.name == 'images':
            root = train_dir.parent.parent
            train_images, train_labels = root / 'images/train', root / 'labels/train'
            split_paths = lambda name: (root / 'images' / name, root / 'labels' / name)
        else:
            root = train_dir.parent
            train_images, train_labels = root / 'train/images', root / 'train/labels'
            split_paths = lambda name: (root / name / 'images', root / name / 'labels')
        if not (train_images.is_dir() and train_labels.is_dir()):
            continue
        for name in ('val', 'valid', 'validation'):
            val_images, val_labels = split_paths(name)
            if val_images.is_dir() and val_labels.is_dir():
                layouts.append((root, train_images, train_labels, val_images, val_labels))
                break
    return layouts

layouts = find_layouts(EXTRACT_DIR)
if len(layouts) != 1:
    raise ValueError(
        f'Expected one YOLO detection dataset with train and val/valid images and labels; '
        f'found {len(layouts)} layouts in {ZIP_PATH.name}.'
    )
DATASET_DIR, train_images, train_labels, val_images, val_labels = layouts[0]
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
train_count = sum(p.suffix.lower() in image_exts for p in train_images.rglob('*'))
val_count = sum(p.suffix.lower() in image_exts for p in val_images.rglob('*'))
label_count = sum(1 for _ in train_labels.rglob('*.txt'))
if not (train_count and val_count and label_count):
    raise ValueError(
        f'Dataset needs train and validation images plus YOLO training labels; '
        f'found train={train_count}, val={val_count}, train labels={label_count}.'
    )

source_yaml = DATASET_DIR / 'data.yaml'
source_config = yaml.safe_load(source_yaml.read_text(encoding='utf-8')) if source_yaml.is_file() else {}
source_config = source_config if isinstance(source_config, dict) else {}
class_names = source_config.get('names', ['object'])
if not isinstance(class_names, (list, dict)):
    class_names = ['object']
DATA_YAML = EXTRACT_DIR / 'colosseum_data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(DATASET_DIR),
    'train': str(train_images.relative_to(DATASET_DIR)),
    'val': str(val_images.relative_to(DATASET_DIR)),
    'names': class_names,
}, sort_keys=False), encoding='utf-8')

os.chdir(OUTPUT_DIR)
print('ZIP:', ZIP_PATH)
print('Extracted dataset:', DATASET_DIR)
print('Images: train=', train_count, 'val=', val_count, '| training labels=', label_count)
print('Classes:', class_names)
print('Dataset config:', DATA_YAML)
print('Results:', OUTPUT_DIR / 'runs' / 'custom')


## 3. Train your model

Set `LABEL` to the object you want to detect. `SOURCE_CLASS` selects its class ID from the ZIP's annotation files; the previous cell prints available class names. Use `GRID = '2x8'` for wide, horizontal objects such as license plates; use `4x4` for more balanced shapes. The default 300 + 20 epochs are the full two-stage recipe; lower them only for a quick pipeline check.

Training writes each seed's `base.pt`, `base_last.pt`, `final.pt`, `final_last.pt`, `colosseum.pt`, and `metrics.json` directly to `MyDrive/Colosseum/runs/custom/RUN_NAME/seedN/`. Checkpoints are saved during training, so completed stages remain in Drive if a later stage is interrupted.

The cell below checks CUDA before a long run and prints the number of images and batches per epoch. The trainer reports an epoch only after training **and validation** for that epoch finish.


In [ ]:
RUN_NAME = 'my_object'
LABEL = 'my object'
GRID = '4x4'
SOURCE_CLASS = 0
SEEDS = (0,)  # Use (0, 1, 2) for a three-seed experiment.
BASE_EPOCHS = 300
SCORE_EPOCHS = 20
BATCH_SIZE = 16

import math
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. In Colab select Runtime > Change runtime type > GPU, then rerun the notebook.')
os.environ['PYTHONUNBUFFERED'] = '1'  # Show subprocess startup messages immediately.
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print(f'Images: train={train_count}, val={val_count}; batches per epoch={math.ceil(train_count / BATCH_SIZE)}', flush=True)
print('The first epoch result appears after one full training and validation pass.', flush=True)

from colosseum import Colosseum
model = Colosseum()
checkpoints = model.train(
    data=DATA_YAML, name=RUN_NAME, label=LABEL, grid=GRID,
    source_class=SOURCE_CLASS, seeds=SEEDS,
    base_epochs=BASE_EPOCHS, score_epochs=SCORE_EPOCHS,
    batch_size=BATCH_SIZE,
)
print('Checkpoints:', *checkpoints, sep='\n- ')


## 4. Read validation metrics

Each seed writes `metrics.json` beside its checkpoint. AP50, AP75 and AP50:95 are computed on the validation split. Precision, recall and F1 use confidence 0.35 and IoU 0.50.


In [ ]:
import json

metric_keys = ('ap50', 'ap75', 'ap50_95', 'precision', 'recall', 'f1')
for checkpoint in checkpoints:
    report = json.loads(checkpoint.with_name('metrics.json').read_text())
    print(f'Seed {report["seed"]} | ' + ' | '.join(
        f'{key}={report[key]:.4f}' for key in metric_keys
    ))
print('First seed metrics are also available as model.metrics')


## 5. Try prediction on one validation image

This is a visual check on an existing validation image. The saved checkpoint can later be loaded with `Colosseum(checkpoints[0])`.


In [ ]:
import yaml
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

config = yaml.safe_load(DATA_YAML.read_text())
base = Path(config.get('path', DATA_YAML.parent))
if not base.is_absolute():
    base = (DATA_YAML.parent / base).resolve()
val_images = base / config['val']
image_paths = sorted(p for p in val_images.rglob('*') if p.suffix.lower() in
                     {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
if not image_paths:
    raise FileNotFoundError(f'No validation images found in {val_images}')
image_path = image_paths[0]
boxes = model.predict(image_path)
with Image.open(image_path) as opened:
    preview = ImageOps.exif_transpose(opened).convert('RGB')
draw = ImageDraw.Draw(preview)
for box in boxes:
    draw.rectangle((box.x1, box.y1, box.x2, box.y2), outline='red', width=3)
    draw.text((box.x1, max(0, box.y1 - 12)),
              f'{box.label} {box.confidence:.2f}', fill='red')
print(image_path, '| detections:', len(boxes))
display(preview)


## 6. Find all saved files in Google Drive

Every `.pt` checkpoint and `metrics.json` is already in your Drive output folder. This cell lists the files for each seed.


In [ ]:
for checkpoint in checkpoints:
    seed_dir = checkpoint.parent
    print(f'\n{seed_dir}')
    for result in sorted(seed_dir.glob('*.pt')):
        print(' -', result.name, f'({result.stat().st_size / 1024 / 1024:.1f} MiB)')
    metrics_path = seed_dir / 'metrics.json'
    if metrics_path.is_file():
        print(' -', metrics_path.name)
print('Google Drive results:', OUTPUT_DIR / 'runs' / 'custom' / RUN_NAME)
